In [3]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

2.7.1+cu128
True


In [6]:
print(torch.tensor([1, 2, 3]))
print(torch.Tensor([[1, 2, 3], [4, 5, 6]]))
print(torch.LongTensor([1, 2, 3]))
print(torch.FloatTensor([1, 2, 3]))


tensor([1, 2, 3])
tensor([[1., 2., 3.],
        [4., 5., 6.]])
tensor([1, 2, 3])
tensor([1., 2., 3.])


In [5]:
tensor = torch.rand(1, 2)
print(tensor.shape)
print(tensor.dtype)
print(tensor.device)

torch.Size([1, 2])
torch.float32
cpu


In [7]:
tensor = tensor.reshape(2,1)
print(tensor)
print(tensor.shape)

tensor([[0.1614],
        [0.0946]])
torch.Size([2, 1])


In [8]:
tensor = torch.rand((3, 3), dtype = torch.float)
print(tensor)

tensor([[0.9080, 0.7857, 0.2824],
        [0.7522, 0.3996, 0.9665],
        [0.1562, 0.2630, 0.8182]])


In [ ]:
if torch.cuda.is_available():
	device = torch.device('cuda')
else:
	device=torch.device('cpu')

print(torch.__version__)
print(device)

print(torch.cuda.is_available())

2.7.1+cu128
cuda
True


In [11]:
cpu = torch.FloatTensor([1, 2, 3])
gpu = torch.cuda.FloatTensor([1, 2, 3])
tensor = torch.rand((1, 1), device = device)
print(cpu)
print(gpu)
print(tensor)

tensor([1., 2., 3.])
tensor([1., 2., 3.], device='cuda:0')
tensor([[0.5520]], device='cuda:0')


C:\Users\KDS-30\AppData\Local\Temp\ipykernel_6580\4063380246.py:2: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\tensor\python_tensor.cpp:80.)
  gpu = torch.cuda.FloatTensor([1, 2, 3])


In [12]:
cpu = torch.FloatTensor([1, 2, 3])
gpu = cpu.cuda()
gpu2cpu = gpu.cpu()
cpu2gpu = cpu.to('cuda')

In [13]:
import numpy as np

ndarray = np.array([1, 2, 3], dtype = np.uint8)
print(torch.tensor(ndarray))
print(torch.Tensor(ndarray))
print(torch.from_numpy(ndarray))

tensor([1, 2, 3], dtype=torch.uint8)
tensor([1., 2., 3.])
tensor([1, 2, 3], dtype=torch.uint8)


In [14]:
tensor = torch.cuda.FloatTensor([1, 2, 3])
ndarray = tensor.detach().cpu().numpy()
print(ndarray)
print(type(ndarray))

[1. 2. 3.]
<class 'numpy.ndarray'>


In [34]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

In [ ]:
class CustomDataset(Dataset):
	def __init__(self, file_path):
		df = pd.read_csv(file_path)
		self.x = df.iloc[:, 0].values
		self.y = df.iloc[:, 1].values
		self.length = len(df)
	def __getitem__(self, index):
		x = torch.FloatTensor([self.x[index] ** 2, self.x[index]])
		y = torch.FloatTensor([self.y[index]])
		return x, y
	def __len__(self):
		return self.length

In [ ]:
class CustomModel(nn.Module):
	def __init__(self):
		super().__init__()
		self.layer = nn.Linear(2, 1)
	def forward(self, x):
		x = self.layer(x)
		return x

In [ ]:
dataset = CustomDataset('./dataset/non_linear.csv')
dataset_size = len(dataset)
train_size = int(dataset_size * 0.8)
val_size = int(dataset_size * 0.1)
test_size = dataset_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(dataset,
														[train_size, val_size, test_size])

train_dataloader = DataLoader(dataset,
							  batch_size = 16,
							  shuffle = True,
							  drop_last = True)
val_dataloader = DataLoader(dataset,
							  batch_size = 4,
							  shuffle = True,
							  drop_last = True)
test_dataloader = DataLoader(dataset,
							  batch_size = 4,
							  shuffle = False,
							  drop_last = True)

In [38]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [39]:
model = CustomModel().to(device)
criterion = nn.MSELoss().to(device)
optimizer = optim.SGD(model.parameters(), lr = 0.0001)

In [ ]:
checkpoint = 1

for epoch in range(1000):
	cost = 0.0
	for x, y in train_dataloader:
		x = x.to(device)
		y = y.to(device)
		output = model(x)
		loss = criterion(output, y)
		optimizer.zero_grad()
		loss.backward()
		optimizer.step()
		cost += loss
	cost = cost / len(train_dataloader)
	if (epoch + 1) % 100 == 0:
		print(epoch + 1, cost)
		torch.save('./models/checkpoint - {checkpoint}.pt')
		checkpoint += 1

100 tensor(0.1008, device='cuda:0', grad_fn=<DivBackward0>)
200 tensor(0.0967, device='cuda:0', grad_fn=<DivBackward0>)
300 tensor(0.0890, device='cuda:0', grad_fn=<DivBackward0>)
400 tensor(0.0885, device='cuda:0', grad_fn=<DivBackward0>)
500 tensor(0.0871, device='cuda:0', grad_fn=<DivBackward0>)
600 tensor(0.0868, device='cuda:0', grad_fn=<DivBackward0>)
700 tensor(0.0838, device='cuda:0', grad_fn=<DivBackward0>)
800 tensor(0.0823, device='cuda:0', grad_fn=<DivBackward0>)
900 tensor(0.0827, device='cuda:0', grad_fn=<DivBackward0>)
1000 tensor(0.0801, device='cuda:0', grad_fn=<DivBackward0>)


In [ ]:
with torch.no_grad():
	model.eval()
	for x, y in val_dataloader:
		x = x.to(device)
		y = y.to(device)      
	#inputs = torch.FloatTensor(
	#    [
	#        [1 ** 2, 1],
	#        [5 ** 2, 5],
	#        [11 ** 2, 11],
	#	]
	#).to(device)
	outputs = model(inputs)
	print(outputs)

tensor([[  1.9728],
        [ 69.5220],
        [356.7533]], device='cuda:0')


In [42]:
torch.save(model, './models/model.pt')

In [43]:
torch.save(model.state_dict(), './models/model_state_dict.pt')

In [ ]:
# CustomModel 클래스 정의 먼저 실행

model = torch.load(
	'./models/model.pt',
	map_location=device,
	weights_only=False
)

model.eval()

CustomModel(
  (layer): Linear(in_features=2, out_features=1, bias=True)
)

In [45]:
print(model)

CustomModel(
  (layer): Linear(in_features=2, out_features=1, bias=True)
)


In [48]:
model_state_dict = torch.load('./models/model_state_dict.pt', map_location = device)
model.load_state_dict(model_state_dict)

<All keys matched successfully>

In [50]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

In [ ]:
class CustomDataset(Dataset):
	def __init__(self, file_path):
		df = pd.read_csv(file_path)
		self.x1 = df.iloc[:, 0].values
		self.x2 = df.iloc[:, 1].values
		self.x3 = df.iloc[:, 2].values
		self.y = df.iloc[:, 3].values
		self.length = len(df)
	def __getitem__(self, index):
		x = torch.FloatTensor([self.x1[index], self.x2[index], self.x3[index]])
		y = torch.FloatTensor([int(self.y[index])])
		return x, y
	def __len__(self):
		return self.length

In [ ]:
class CustomModel(nn.Module):
	def __init__(self):
		super().__init__()
		self.layer = nn.Sequential(
			nn.Linear(3, 1),
			nn.Sigmoid()
		)

	def forward(self, x):
		x = self.layer(x)
		return(x)

In [ ]:
dataset = CustomDataset('./dataset/binary.csv')
dataset_size = len(dataset)
train_size = int(dataset_size * 0.8)
val_size = int(dataset_size * 0.1)
test_size = dataset_size - train_size - val_size

train_dataloader, val_dataloader, test_dataloader = random_split(dataset,
																 [train_size, val_size, test_size],
																 torch.manual_seed(42))

train_dataloader = DataLoader(dataset,
							  batch_size = 64,
							  shuffle = True,
							  drop_last = True)
val_dataloader = DataLoader(dataset,
							  batch_size = 4,
							  shuffle = True,
							  drop_last = True)
test_dataloader = DataLoader(dataset,
							  batch_size = 4,
							  shuffle = False,
							  drop_last = True)


In [54]:
model = CustomModel().to(device)
criterion = nn.BCELoss().to(device)
optimizer = optim.SGD(model.parameters(), lr = 0.0001)

In [ ]:
for epoch in range(1000):
	cost = 0.0
	for x, y in train_dataloader:
		x = x.to(device)
		y = y.to(device)
		output = model(x)
		loss = criterion(output, y)
		optimizer.zero_grad()
		loss.backward()
		optimizer.step()
	cost = cost / len(train_dataloader)
	if (epoch + 1) % 100 == 0:
		print(epoch, cost)

99 0.0
199 0.0
299 0.0
399 0.0
499 0.0
599 0.0
699 0.0
799 0.0
899 0.0
999 0.0


In [ ]:
with torch.no_grad():
	model.eval()
	for x, y in val_dataloader:
		x = x.to(device)
		y = y.to(device)
		outputs = model(x)
		print(outputs)
		print(outputs >= torch.FloatTensor([0.5]).to(device))

tensor([[0.5401],
        [0.5915],
        [0.5926],
        [0.6456]], device='cuda:0')
tensor([[True],
        [True],
        [True],
        [True]], device='cuda:0')
tensor([[0.5405],
        [0.6197],
        [0.5707],
        [0.5963]], device='cuda:0')
tensor([[True],
        [True],
        [True],
        [True]], device='cuda:0')
tensor([[0.6214],
        [0.6265],
        [0.5783],
        [0.6185]], device='cuda:0')
tensor([[True],
        [True],
        [True],
        [True]], device='cuda:0')
tensor([[0.6392],
        [0.4491],
        [0.6407],
        [0.5825]], device='cuda:0')
tensor([[ True],
        [False],
        [ True],
        [ True]], device='cuda:0')
tensor([[0.5485],
        [0.6257],
        [0.4178],
        [0.6343]], device='cuda:0')
tensor([[ True],
        [ True],
        [False],
        [ True]], device='cuda:0')
tensor([[0.6579],
        [0.5872],
        [0.5745],
        [0.6625]], device='cuda:0')
tensor([[True],
        [True],
        [T

In [1]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [ ]:
import torch.nn as nn
class Net(nn.Module):
	def __init__(self):
		super().__init__()
		self.layer = nn.Sequential(
			nn.Linear(1, 2),
			nn.Sigmoid()
		)
		self.fc = nn.Linear(2, 1)
		self._init_weights()

	def _init_weights(self):
		nn.init.xavier_uniform_(self.layer[0].weight)
		self.layer[0].bias.data.fill_(0.01)

		nn.init.xavier_uniform_(self.fc.weight)
		self.fc.bias.data.fill_(0.01)

model = Net().to(device)

In [ ]:
for name, param in model.named_parameters():
	print(name, param.data)

layer.0.weight tensor([[ 1.0315],
        [-0.5469]], device='cuda:0')
layer.0.bias tensor([0.0100, 0.0100], device='cuda:0')
fc.weight tensor([[1.1987, 0.4027]], device='cuda:0')
fc.bias tensor([0.0100], device='cuda:0')


In [ ]:
class Net(nn.Module):
	def __init__(self):
		super().__init__()
		self.layer = nn.Sequential(
			nn.Linear(1, 2),
			nn.Sigmoid()
		)
		self.fc = nn.Linear(2, 1)
		self.apply(self._init_weights)

	def _init_weights(self, module):
		if isinstance(module, nn.Linear):
			nn.init.xavier_uniform_(module.weight)
			nn.init.constant_(module.bias, 0.01)

		print(f'Apply : {module}')

model = Net().to(device)

Apply : Linear(in_features=1, out_features=2, bias=True)
Apply : Sigmoid()
Apply : Sequential(
  (0): Linear(in_features=1, out_features=2, bias=True)
  (1): Sigmoid()
)
Apply : Linear(in_features=2, out_features=1, bias=True)
Apply : Net(
  (layer): Sequential(
    (0): Linear(in_features=1, out_features=2, bias=True)
    (1): Sigmoid()
  )
  (fc): Linear(in_features=2, out_features=1, bias=True)
)


In [ ]:
for name, param in model.named_parameters():
	print(name, param.data)

layer.0.weight tensor([[1.1412],
        [0.4128]], device='cuda:0')
layer.0.bias tensor([0.0100, 0.0100], device='cuda:0')
fc.weight tensor([[ 0.4023, -0.4705]], device='cuda:0')
fc.bias tensor([0.0100], device='cuda:0')


In [ ]:
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
x_data = torch.rand(256, 1) * 10
y_data = 2 * x_data + 1 + torch.randn(256, 1) * 0.5
train_dataset = TensorDataset(x_data, y_data)
train_dataloader = DataLoader(train_dataset,
							  batch_size = 32,
							  shuffle = True,
							  drop_last = True)
model = nn.Linear(1, 1).to(device)
criterion = nn.MSELoss().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr = 0.001)


In [ ]:
for epoch in range(10):
	for x, y in train_dataloader:
		x = x.to(device)
		y = y.to(device)

		output = model(x)
		_lambda = 0.5
		#l1_loss = sum(p.abs().sum() for p in model.parameters())
		l2_loss = sum(p.pow(2.0).sum() for p in model.parameters())

		#loss = criterion(output, y) + _lambda * l1_loss
		loss = criterion(output, y) + _lambda * l2_loss
		optimizer.zero_grad()
		loss.backward()
		optimizer.step()
	#print(epoch + 1, loss.item(), l1_loss.item())
	print(epoch + 1, loss.item(), l2_loss.item())

1 2.9995105266571045 4.898513317108154
2 3.2556302547454834 4.861276626586914
3 3.2960360050201416 4.839059829711914
4 3.112140655517578 4.819094657897949
5 3.484954595565796 4.7994384765625
6 3.2294023036956787 4.782095909118652
7 2.812739133834839 4.7709503173828125
8 3.527942657470703 4.755488395690918
9 2.9947705268859863 4.747015953063965
10 2.9083685874938965 4.73085880279541


In [12]:
print(model.weight.item(), model.bias.item()) 

2.1588799953460693 -0.2572735846042633


In [ ]:
# model = nn.Linear(1, 1).to(device)
# optimizer = torch.optim.SGD(model.parameters(),
#                             lr = 0.01,
#                             weight_decay = 0.01)

In [ ]:
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
x_data = torch.rand(256, 1) * 10
y_data = 2 * x_data + 1 + torch.randn(256, 1) * 0.5
train_dataset = TensorDataset(x_data, y_data)
train_dataloader = DataLoader(train_dataset,
							  batch_size = 32,
							  shuffle = True,
							  drop_last = True)
model = nn.Linear(1, 1).to(device)
criterion = nn.MSELoss().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr = 0.001)

In [ ]:
for epoch in range(20):
	for x, y in train_dataloader:
		x = x.to(device)
		y = y.to(device)

		output = model(x)
		loss = criterion(output, y)

		optimizer.zero_grad()
		loss.backward()

		torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)

		optimizer.step()
	print(epoch, loss.item())	

In [15]:
import nltk

In [18]:
for resource in ['wordnet', 'omw-1.4', 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng']:
    nltk.download(resource, quiet = True)

In [19]:
import nlpaug.augmenter.word as naw

texts = [
    'Those who can imagine anything, can create the impossible.',
    'We can only see a short distance ahead, but we can see plenty there that needs to be done.',
    'If a machine is expected to be infallible, it cannot also be intelligent'
]

aug = naw.ContextualWordEmbsAug(model_path = 'bert-base-uncased', action = 'insert')
augmented_texts = aug.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(f'src : {text}')
    print(f'dst : {augmented}')
    print('-------------------------')

c:\Users\KDS-30\Desktop\class\10_LLM_CV\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


src : Those who can imagine anything, can create the impossible.
dst : only those who say can imagine anything, i can create seemingly the impossible.
-------------------------
src : We can only see a short distance ahead, but we can see plenty there that needs to be done.
dst : thus we now can only just see a short distance ahead, individually but we really can see about plenty there that needs to quickly be done.
-------------------------
src : If a machine is expected to be infallible, it cannot also be intelligent
dst : however if then a machine c is expected to be totally infallible, that it cannot also be intelligent
-------------------------


In [21]:
import nlpaug.augmenter.char as nac

texts = [
    'Those who can imagine anything, can create the impossible.',
    'We can only see a short distance ahead, but we can see plenty there that needs to be done.',
    'If a machine is expected to be infallible, it cannot also be intelligent'
]

aug = nac.RandomCharAug(action = 'delete')
augmented_texts = aug.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(f'src : {text}')
    print(f'dst : {augmented}')
    print('-------------------------')

src : Those who can imagine anything, can create the impossible.
dst : Tse who can imagine anthi, can cete the imosble.
-------------------------
src : We can only see a short distance ahead, but we can see plenty there that needs to be done.
dst : We can on see a srt distance hea, but we can see pley ter tt needs to be dn.
-------------------------
src : If a machine is expected to be infallible, it cannot also be intelligent
dst : If a maci is execd to be infalbe, it cann lo be intelligent
-------------------------


In [22]:
import nlpaug.augmenter.word as naw

texts = [
    'Those who can imagine anything, can create the impossible.',
    'We can only see a short distance ahead, but we can see plenty there that needs to be done.',
    'If a machine is expected to be infallible, it cannot also be intelligent'
]

aug = naw.RandomWordAug(action = 'swap')
augmented_texts = aug.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(f'src : {text}')
    print(f'dst : {augmented}')
    print('-------------------------')

src : Those who can imagine anything, can create the impossible.
dst : Imagine those who can anything, can the create impossible.
-------------------------
src : We can only see a short distance ahead, but we can see plenty there that needs to be done.
dst : We can only see a short distance ahead, but can we plenty see there needs that to be done.
-------------------------
src : If a machine is expected to be infallible, it cannot also be intelligent
dst : Machine if a is expected to be infallible it, cannot be intelligent also
-------------------------


In [24]:
import nlpaug.augmenter.word as naw

texts = [
    'Those who can imagine anything, can create the impossible.',
    'We can only see a short distance ahead, but we can see plenty there that needs to be done.',
    'If a machine is expected to be infallible, it cannot also be intelligent'
]

aug = naw.SynonymAug(aug_src = 'wordnet')
augmented_texts = aug.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(f'src : {text}')
    print(f'dst : {augmented}')
    print('-------------------------')

src : Those who can imagine anything, can create the impossible.
dst : Those who terminate imagine anything, pot create the inconceivable.
-------------------------
src : We can only see a short distance ahead, but we can see plenty there that needs to be done.
dst : We tail just consider a short distance beforehand, but we lavatory see plenty on that point that needs to be arrange.
-------------------------
src : If a machine is expected to be infallible, it cannot also be intelligent
dst : If a auto is expect to be infallible, it cannot also be sound
-------------------------


In [26]:
import nlpaug.augmenter.word as naw

texts = [
    'Those who can imagine anything, can create the impossible.',
    'We can only see a short distance ahead, but we can see plenty there that needs to be done.',
    'If a machine is expected to be infallible, it cannot also be intelligent'
]

reserved_tokens = [
    ["can", "can't", "cannot", "could"]
]

reserved_aug = naw.ReservedAug(reserved_tokens = reserved_tokens)
augmented_texts = reserved_aug.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(f'src : {text}')
    print(f'dst : {augmented}')
    print('-------------------------')

src : Those who can imagine anything, can create the impossible.
dst : Those who can't imagine anything, can't create the impossible.
-------------------------
src : We can only see a short distance ahead, but we can see plenty there that needs to be done.
dst : We cannot only see a short distance ahead, but we can't see plenty there that needs to be done.
-------------------------
src : If a machine is expected to be infallible, it cannot also be intelligent
dst : If a machine is expected to be infallible, it can also be intelligent
-------------------------


In [29]:
import nlpaug.augmenter.word as naw

texts = [
    'Those who can imagine anything, can create the impossible.',
    'We can only see a short distance ahead, but we can see plenty there that needs to be done.',
    'If a machine is expected to be infallible, it cannot also be intelligent'
]

back_translation = naw.BackTranslationAug(
    from_model_name = "facebook/wmt19-en-de",
    to_model_name = "facebook/wmt19-de-en",
    device = 'cpu'
)
augmented_texts = back_translation.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(f'src : {text}')
    print(f'dst : {augmented}')
    print('-------------------------')

src : Those who can imagine anything, can create the impossible.
dst : Anyone who can imagine anything can achieve the impossible.
-------------------------
src : We can only see a short distance ahead, but we can see plenty there that needs to be done.
dst : We can only look a little ahead, but we can see a lot there that needs to be done.
-------------------------
src : If a machine is expected to be infallible, it cannot also be intelligent
dst : If a machine is expected to be infallible, it cannot be intelligent either
-------------------------
